# Immune sub-split

The immune compartment is too heterogeneous to annotate in one pass, so it is
split again before labelling - into lymphocytes, granulocytes and macrophages.
Each of those is then cleaned and annotated on its own, and the three are
recombined afterwards.

The sub-grouping is recorded in `obs['immune_compartment']`, which stays on the
cells through to the final object.

Two clustering passes, same shape as the other compartments:

1. **Cluster, inspect, remove junk.**
2. **Re-cluster what remains, then assign the three lineages.**

Set `DATA` in `config.py` before running.

In [ ]:
import scanpy as sc

from config import DATA

SEED = 0

## 1. Load the immune compartment

Written by `compartments.ipynb`.

In [ ]:
adata = sc.read_h5ad(DATA / 'immune.h5ad')
print(adata.shape)

## 2. First pass - cluster to find the junk

> **The resolution below is not a principled value.** It is tuned by eye, and
> the only thing it has to achieve is putting the cells you want to remove into
> a cluster of their own - not split across several, and not mixed in with a
> real population. Raise it until the junk separates cleanly, and no further.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

# tune this by eye until the junk sits in its own cluster - see above
sc.tl.leiden(adata, resolution=0.5, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', palette='tab20', legend_loc='on data',
           frameon=False)

Cells from other compartments that survived the first split show up here. This
is where they get caught.

In [ ]:
sc.pl.umap(adata, color=['Pecam1', 'Pdgfra', 'Pdgfrb', 'Epcam', 'Acta2'],
           frameon=False)

## 3. Drop the junk clusters

> **Cluster numbers here belong to the first pass only.** Re-running this
> notebook renumbers them. Read the plots and rebuild this list.

In [ ]:
JUNK = ['12']

n_before = adata.n_obs
adata = adata[~adata.obs['leiden'].isin(JUNK)].copy()

print(f'{n_before:,} -> {adata.n_obs:,} cells ({n_before - adata.n_obs:,} dropped)')

## 4. Second pass - re-cluster what is left

`resolution = 0.1` - the same coarse value used to find the compartments
themselves, and for the same reason. All this pass has to do is separate three
megagroups: lymphocytes, granulocytes and macrophages. They are far enough
apart that a low resolution finds them cleanly, and anything higher starts
splitting each lineage internally, which only gets in the way here.

**No annotation happens in this notebook.** The three objects are written out
so each can be re-clustered at a resolution suited to it, and `cell_type` is
assigned there.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

sc.tl.leiden(adata, resolution=0.1, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', palette='tab20', legend_loc='on data',
           frameon=False)

## 5. Markers

| Lineage | Markers |
| --- | --- |
| Lymphocytes | `Cd3e` (T), `Nkg7` (NK), `Cd19` (B), `Xbp1` (plasma) |
| Granulocytes | `S100a9`, `Csf1` |
| Macrophages | `Marco`, `Ccl2`, `Spp1`, `Cd74`, `Atox1` |

The lymphocyte panel spans four lineages on purpose - T, NK, B and plasma cells
all belong in the same object at this stage and are separated later.

In [ ]:
sc.pl.umap(adata, color=['Cd3e', 'Nkg7', 'Cd19', 'Xbp1'], frameon=False)  # lymphocytes

In [ ]:
sc.pl.umap(adata, color=['S100a9', 'Csf1'], frameon=False)  # granulocytes

In [ ]:
sc.pl.umap(adata, color=['Marco', 'Ccl2', 'Spp1', 'Cd74', 'Atox1'],
           frameon=False)  # macrophages

## 6. Assign the three lineages

Read off the marker plots above.

> These numbers belong to the second clustering pass in this notebook. They are
> a record of the run that produced the deposited object, not something to
> trust after a re-run.

Anything matching none of the three stays `other`.

In [ ]:
IMMUNE_COMPARTMENTS = {
    'Lymphocytes':  ['0', '2', '6', '8'],
    'Granulocytes': ['3', '9'],
    'Macrophages':  ['1', '4', '5', '7', '10'],
}

adata.obs['immune_compartment'] = 'other'
for name, clusters in IMMUNE_COMPARTMENTS.items():
    adata.obs.loc[adata.obs['leiden'].isin(clusters), 'immune_compartment'] = name

adata.obs['immune_compartment'] = adata.obs['immune_compartment'].astype('category')

print(adata.obs['immune_compartment'].value_counts().to_string())

In [ ]:
sc.pl.umap(adata, color='immune_compartment', frameon=False)

## 7. Write one object per lineage

Each goes on to its own annotation notebook. The full immune object is kept
alongside them, carrying `immune_compartment` for every cell.

In [ ]:
for name in IMMUNE_COMPARTMENTS:
    sub = adata[adata.obs['immune_compartment'] == name].copy()

    out = DATA / f'{name.lower()}.h5ad'
    sub.write_h5ad(out)
    print(f'{out.name}: {sub.n_obs:,} cells')

In [ ]:
adata.write_h5ad(DATA / 'immune_split.h5ad')
print(f'{adata.n_obs:,} cells written')